# Lab: Convolutional Neural Networks for Satellite Image Classification

**Instructor:** Muhammad Sayed  
**Semester:** Spring 2026

---

### Learning Objectives
* A custom **Convolutional Neural Network** architecture is designed and implemented using PyTorch.
* **High-resolution satellite imagery** is mapped from fine-grained categories to broad macro-classes.
* **Classification model generalization** is evaluated through systematic **hyperparameter tuning**.
* Classification errors are analyzed by **tracing misclassified macro-classes back to their original micro-class spectral signatures**.
* Results and experimental trials are serialized utilizing Pydantic models for **standardized** evaluation.


In [2]:
import os
import json
from enum import Enum
from typing import List, Dict, Any
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, Dataset, WeightedRandomSampler
from torchvision.datasets import ImageFolder
from torchvision import transforms
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from pydantic import BaseModel
import warnings

from collections import Counter

warnings.filterwarnings('ignore')

### Data Engineering and Macro-Class Mapping
The required NWPU-RESISC45 dataset can be downloaded from [Kaggle](https://www.kaggle.com/datasets/aqibrehmanpirzada/nwpuresisc45). It consists of forty-five fine-grained classes. These are algorithmically mapped to four broad macro-classes. Cloud cover is categorized as unknown and must be filtered out during dataset initialization.

In [15]:
class LandCover(Enum):
    UNKNOWN = 0
    GREENERY = 1
    SAND = 2
    WATER = 3
    CEMENT_INFRASTRUCTURE = 4

macro_class_mapping = {
    'chaparral': LandCover.GREENERY, 
    'circular_farmland': LandCover.GREENERY, 
    'forest': LandCover.GREENERY,
    'golf_course': LandCover.GREENERY, 
    'meadow': LandCover.GREENERY, 
    'rectangular_farmland': LandCover.GREENERY, 
    'terrace': LandCover.GREENERY,

    'beach': LandCover.SAND, 
    'desert': LandCover.SAND, 
    'mountain': LandCover.SAND, 
    'island': LandCover.SAND,

    'harbor': LandCover.WATER, 
    'lake': LandCover.WATER, 
    'river': LandCover.WATER, 
    'sea_ice': LandCover.WATER, 
    'wetland': LandCover.WATER, 
    'ship': LandCover.WATER, 
    'snowberg': LandCover.WATER,

    'airport': LandCover.CEMENT_INFRASTRUCTURE, 
    'bridge': LandCover.CEMENT_INFRASTRUCTURE, 
    'church': LandCover.CEMENT_INFRASTRUCTURE, 
    'commercial_area': LandCover.CEMENT_INFRASTRUCTURE,
    'dense_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'freeway': LandCover.CEMENT_INFRASTRUCTURE, 
    'industrial_area': LandCover.CEMENT_INFRASTRUCTURE,
    'intersection': LandCover.CEMENT_INFRASTRUCTURE, 
    'medium_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'mobile_home_park': LandCover.CEMENT_INFRASTRUCTURE,
    'overpass': LandCover.CEMENT_INFRASTRUCTURE, 
    'palace': LandCover.CEMENT_INFRASTRUCTURE, 
    'parking_lot': LandCover.CEMENT_INFRASTRUCTURE, 
    'railway': LandCover.CEMENT_INFRASTRUCTURE,
    'railway_station': LandCover.CEMENT_INFRASTRUCTURE, 
    'roundabout': LandCover.CEMENT_INFRASTRUCTURE, 
    'runway': LandCover.CEMENT_INFRASTRUCTURE,
    'sparse_residential': LandCover.CEMENT_INFRASTRUCTURE, 
    'storage_tank': LandCover.CEMENT_INFRASTRUCTURE, 
    'thermal_power_station': LandCover.CEMENT_INFRASTRUCTURE,
    'baseball_diamond': LandCover.CEMENT_INFRASTRUCTURE, 
    'basketball_court': LandCover.CEMENT_INFRASTRUCTURE, 
    'ground_track_field': LandCover.CEMENT_INFRASTRUCTURE,
    'stadium': LandCover.CEMENT_INFRASTRUCTURE, 
    'tennis_court': LandCover.CEMENT_INFRASTRUCTURE, 
    'airplane': LandCover.CEMENT_INFRASTRUCTURE,

    'cloud': LandCover.UNKNOWN
}

### Dataset Initialization and Loading
The provided dataset directory is natively partitioned into training and testing subsets. Images must be loaded and mapped to their respective macro-classes, ensuring instances mapped to the `UNKNOWN` category are explicitly excluded. The testing subset is to be preserved strictly for final evaluation. To facilitate hyperparameter tuning without data leakage, the training subset must be further partitioned to generate an independent validation subset, or a cross-validation strategy must be implemented.

In [16]:
# A transform pipeline is required (e.g., resizing, tensor conversion, normalization).
# train_dataset = ImageFolder(root='path_to_train_dir', transform=...)
# test_dataset = ImageFolder(root='path_to_test_dir', transform=...)

transform = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize([0.5] * 3, [0.5] * 3),
    ]
)

train_dataset = ImageFolder(root='Dataset/train', transform=transform)
test_dataset = ImageFolder(root="Dataset/test", transform=transform)

# Valid indices and mapped targets are to be extracted for both datasets, excluding the UNKNOWN class.
# The train_dataset must be dynamically split into distinct training and validation subsets.
# DataLoaders for the training, validation, and testing subsets are to be instantiated here.
def map_to_macro(dataset):
    indices = []
    labels = []

    for i, (path, _) in enumerate(dataset.samples):
        class_name = path.split(os.sep)[-2]
        macro = macro_class_mapping[class_name]

        if macro != LandCover.UNKNOWN:
            indices.append(i)
            labels.append(macro.value - 1)

    return indices, labels

train_indices, train_labels = map_to_macro(train_dataset)
test_indices, test_labels = map_to_macro(test_dataset)


In [17]:
class MacroDataset(Dataset):
    def __init__(self, dataset, indices, labels):
        self.dataset = dataset
        self.indices = indices
        self.labels = labels

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        image, _ = self.dataset[real_idx]
        label = self.labels[idx]
        return image, label

In [18]:
val_split = 0.2
num_train = len(train_indices)
indices = list(range(num_train))
np.random.shuffle(indices)

split = int(val_split * num_train)
train_idx = indices[split:]
val_idx = indices[:split]

# Convert subset positions back to real indices of the filtered dataset.
train_subset_indices = [train_indices[i] for i in train_idx]
val_subset_indices = [train_indices[i] for i in val_idx]

train_data = MacroDataset(
    train_dataset, train_subset_indices, [train_labels[i] for i in train_idx]
)

val_data = MacroDataset(
    train_dataset, val_subset_indices, [train_labels[i] for i in val_idx]
)
test_data = MacroDataset(test_dataset, test_indices, test_labels)

### Convolutional Neural Network Architecture
A custom CNN class is defined. Spatial dimensions must be carefully tracked between convolutional outputs and fully connected layer inputs.

As a reference model, the LeNet-5 architecture illustrates the foundational principles of convolutional networks. The input tensor is sequentially processed through alternating convolutional and pooling layers to extract spatial features and reduce dimensionality. The resulting feature maps are subsequently flattened and passed through fully connected layers to yield the final classification probabilities.


In [19]:
class SatelliteCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1), # 256x256x32
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), # 128x128x32

            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 128x128x64
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), # 64x64x64

            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 64x64x128
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), # 32x32x128

            nn.Conv2d(128, 256, kernel_size=3, padding=1), # 32x32x256
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2), # 16x16x256

            nn.Dropout2d(0.2) # 
        )
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1)) # 1x1x256
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.avg_pool(x)
        x = self.classifier(x)
        return x

### Model Training
The loss function and optimizer are initialized. The training loop is executed, updating model weights while validating performance on the held-out validation set to monitor for overfitting.

In [20]:
# Model, Loss Criterion, and Optimizer are instantiated here.
# The training loop over defined epochs is executed here.
def train_model(model, train_loader, val_loader, optimizer, epochs):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    criterion = nn.CrossEntropyLoss()

    model.to(device)
    
    best_val_acc = 0.0
    best_model_state = None

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        # Validation
        model.eval()
        val_preds, val_true = [], []

        with torch.no_grad():
            for images, labels in val_loader:
                images = images.to(device)
                outputs = model(images)
                preds = torch.argmax(outputs, dim=1).cpu().numpy()

                val_preds.extend(preds)
                val_true.extend(labels.numpy())

        val_acc = accuracy_score(val_true, val_preds)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = model.state_dict().copy()

        print(f"Epoch {epoch+1}: Loss={total_loss/len(val_loader):.3f}, ValAcc={val_acc:.3f}")
    
    # Load best model state
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model

### Quantitative Evaluation
The trained model is evaluated against the test subset. Overall Accuracy, Macro-Averaged F1 Scores, and the Confusion Matrix are computed.

In [21]:
# Predictions on the test set are generated.
# accuracy_score, f1_score, and confusion_matrix are computed and printed.
def evaluate(model, loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            out = model(x)
            preds = torch.argmax(out, dim=1).cpu().numpy()

            y_pred.extend(preds)
            y_true.extend(y.numpy())

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average=None)
    cm = confusion_matrix(y_true, y_pred)

    return acc, f1, cm, y_true, y_pred

### Hyperparameter Tuning Analysis
Observations regarding hyperparameter tuning are required. Multiple trials must be recorded, documenting the chosen parameters, resultant accuracy, class-specific F-scores, and drawn conclusions. These records are to be serialized using the provided Pydantic schema in the submission cell.


In [36]:
import pickle

trial_records = []

with open("models/trial_records.pkl", "rb") as f:
    trial_records = pickle.load(f)

In [ ]:
# Hyperparameter tuning execution is to be implemented here.
# Utilizing a grid search strategy (e.g., via itertools.product or sklearn.model_selection.ParameterGrid) is suggested to systematically explore combinations of learning rates, batch sizes, optimizers, and epochs.
from itertools import product

param_grid = {
    "lr": [0.01, 0.001],
    "batch_size": [32, 64],
    "optimizer": ["Adam", "SGD"],
    "imbalance_strategy": ["none", "oversampling"] 
}

epochs = 10

# Build weights from the training subset labels so sampler indices match train_data.

trial_num = 12
for lr, bs, opt_name, imbalance_strategy in product(*param_grid.values()):
    trial_num += 1
    print(f"\nTrial {trial_num}: lr={lr}, opt={opt_name}, batch_size={bs}, epochs={epochs}, imbalance_strategy={imbalance_strategy}")

    if imbalance_strategy == "oversampling":
        subset_counts = Counter(train_data.labels)
        sample_weights = [1.0 / subset_counts[label] for label in train_data.labels]
        sampler = WeightedRandomSampler(
            weights=sample_weights,
            num_samples=len(sample_weights),
            replacement=True
        )
        train_loader = DataLoader(train_data, batch_size=bs, sampler=sampler)
    else:
        train_loader = DataLoader(train_data, batch_size=bs, shuffle=True)

    val_loader = DataLoader(val_data, batch_size=bs)
    test_loader = DataLoader(test_data, batch_size=bs)

    model = SatelliteCNN()

    optimizer = (
        optim.Adam(model.parameters(), lr=lr)
        if opt_name == "Adam"
        else optim.SGD(model.parameters(), lr=lr, momentum=0.9)
    )

    model = train_model(model, train_loader, val_loader, optimizer, epochs)
    
    # Save best model for this trial
    model_path = f"best_model_trial_{trial_num}.pth"
    torch.save(model.state_dict(), model_path)
    print(f"Best model saved to {model_path}")

    acc, f1, cm, y_true, y_pred = evaluate(model, test_loader)
    print(f"Test Accuracy: {acc:.4f}")

    trial_records.append(
        {
            "hyperparams": {
                "learning_rate": lr,
                "batch_size": bs,
                "optimizer": opt_name,
                "epochs": epochs,
                "imbalance_strategy": imbalance_strategy,
                "with_dropout_after_features": True
            },
            "accuracy": acc,
            "f1": f1,
            "cm": cm,
            "y_true": y_true,
            "y_pred": y_pred,
            "model_path": model_path,
        }
    )

In [34]:
import pickle 

with open("trial_records.pkl", "wb") as f:
    pickle.dump(trial_records, f)

### Error Analysis and Confusion Patterns
The macro-class exhibiting the lowest accuracy must be identified based on the computed F-scores and Confusion Matrix. Subsequently, the original, fine-grained micro-classes constituting the misclassified instances within this macro-class are to be examined. Patterns explaining why the convolutional neural network struggles to differentiate these specific land cover categories must be formulated and reported in the submission payload.


In [37]:
# Error analysis for the least accurate macro-class.
macro_names = ["GREENERY", "SAND", "WATER", "CEMENT"]

best_trial = max(trial_records, key=lambda x: x["accuracy"])
f1_scores = np.asarray(best_trial["f1"])
worst_class_idx = int(np.argmin(f1_scores))
worst_class = macro_names[worst_class_idx]

print("Worst class name:", worst_class)
print("Per-class F1:", {name: round(float(score), 4) for name, score in zip(macro_names, f1_scores)})

Worst class name: WATER
Per-class F1: {'GREENERY': 0.8674, 'SAND': 0.8558, 'WATER': 0.8283, 'CEMENT': 0.9406}


In [38]:
# Water Analysis 
from pathlib import Path

# Macro-level confusion for the worst class.
cm = np.asarray(best_trial["cm"])
row = cm[worst_class_idx]

print(f"{' ':<15}" + " ".join(f"{name:<15}" for name in macro_names))
print("-" * 75)
print(f"{worst_class:<15}" + " ".join(f"{int(count):<15}" for count in row))
print("-" * 75)


misclassified_sampled = []
for i, (true, predicted) in enumerate(zip(best_trial["y_true"], best_trial["y_pred"])):
    if true == worst_class_idx and predicted != true:
        image_path = Path(test_dataset.samples[test_indices[i]][0])
        true_micro = image_path.parent.name
        pred_macro = macro_names[int(predicted)]
        misclassified_sampled.append((true_micro, pred_macro))

# Build cross-tab manually: true micro-class -> predicted macro-class counts.
micro_to_macro_counts = {}
for true_micro, pred_macro in misclassified_sampled:
    if true_micro not in micro_to_macro_counts:
        micro_to_macro_counts[true_micro] = Counter()
    micro_to_macro_counts[true_micro][pred_macro] += 1


print("\n")


pred_cols = [name for name in macro_names if name != macro_names[worst_class_idx]]
header = f"{'micro':<28}" + " ".join(f"{col:<10}" for col in pred_cols)
print(header)
print("-" * len(header))

for micro in sorted(micro_to_macro_counts):
    row_counts = micro_to_macro_counts[micro]
    row = f"{micro:<28}" + " ".join(f"{row_counts.get(col, 0):<10}" for col in pred_cols)
    print(row)

print("\nMost common (true micro, predicted macro) confusion pairs:")
print(Counter(misclassified_sampled).most_common(9))

               GREENERY        SAND            WATER           CEMENT         
---------------------------------------------------------------------------
WATER          24              23              598             55             
---------------------------------------------------------------------------


micro                       GREENERY   SAND       CEMENT    
------------------------------------------------------------
harbor                      1          0          10        
lake                        3          1          3         
river                       12         7          4         
sea_ice                     0          1          0         
ship                        3          7          29        
snowberg                    1          2          8         
wetland                     4          5          1         

Most common (true micro, predicted macro) confusion pairs:
[(('ship', 'CEMENT'), 29), (('river', 'GREENERY'), 12), (('harbor', 'CEMENT'), 

In [72]:
interpretation = """
# Error Analysis Interpretation

#### The worst macro-class is WATER
#### At the Macro Level, WATER macro class is mostly confused with CEMENT, GREENRY then SAND 
#### At the Micro Level, the most missclassified categories are 

Most common (true micro, predicted macro) 

- [('ship', 'CEMENT'), 29] ship interpreted as CEMENT which make sense because it contains many man-made artificats 
- [('river', 'GREENERY'), 12] river interpreted as GREENRY which make sense because of the plants in the river 
- [('harbor', 'CEMENT'), 10] harbor interpreted as CEMENT which also make sense because it contain many man made structures beside the water
- [('snowberg', 'CEMENT'), 8] snowberg appearing as CEMENT because it mostly look building tops
- [('river', 'SAND'), 7] river interpreted as SAND because rivers are mostly surrounded by soil
- [('ship', 'SAND'), 7] ship interpreted as SAND because it mostly appears near coastal areas 

"""

### Pydantic Models and Submission Export
The `pydantic` library is utilized to ensure structural integrity of the final submission. The models below define the exact schema expected for automated grading. The variables are to be populated with the experimental findings, and the cell executed to write the JSON output.

In [41]:
class TrialRecord(BaseModel):
    hyperparams: Dict[str, Any]
    accuracy: float
    F_score_class: Dict[str, float]
    observations: List[str]
    conclusions: List[str]

trials = []

for record in trial_records:  # Example: using each trial record for demonstration
    trials.append(TrialRecord(
        hyperparams=record["hyperparams"],
        accuracy=record["accuracy"],
        F_score_class={macro_names[i]: record["f1"][i] for i in range(len(macro_names))},
        observations=[""],
        conclusions=[""]
    ))

In [54]:
trials = sorted(trials, key=lambda x: x.accuracy)

In [83]:
for trial in trials:
    print("accuracy:", trial.accuracy)
    print("Hyperparameters:", trial.hyperparams)
    print("F1 scores:", trial.F_score_class)
    print("Observations:", trial.observations)
    print("Conclusions:", trial.conclusions)
    print("=" * 50)

accuracy: 0.8152272727272727
Hyperparameters: {'learning_rate': 0.001, 'batch_size': 32, 'optimizer': 'SGD', 'epochs': 10, 'imbalance_strategy': 'oversampling'}
F1 scores: {'GREENERY': 0.7726971504307488, 'SAND': 0.7001862197392924, 'WATER': 0.715258855585831, 'CEMENT': 0.8856601389766267}
Observations: ['All classes have relatively low F1 scores']
Conclusions: ['The learning rate of 0.001 combined with SGD caused underfitting']
accuracy: 0.8438636363636364
Hyperparameters: {'learning_rate': 0.01, 'batch_size': 64, 'optimizer': 'SGD', 'epochs': 10, 'imbalance_strategy': 'oversampling'}
F1 scores: {'GREENERY': 0.8354814253222138, 'SAND': 0.8226950354609929, 'WATER': 0.7044943820224719, 'CEMENT': 0.9009268795056643}
Observations: ['Increasing the batch size and using a higher learning rate improved accuracy overall but WATER accuracy got lower']
Conclusions: ['The combination of a higher learning rate and larger batch size helped the model converge better but may have caused overfitting 

In [56]:
trials[0].observations = [
    "All classes have relatively low F1 scores"
]

trials[0].conclusions = [
    "The learning rate of 0.001 combined with SGD caused underfitting"
]

In [58]:
trials[1].observations = [
    "Increasing the batch size and using a higher learning rate improved accuracy overall but WATER accuracy got lower"
]

trials[1].conclusions = [
    "The combination of a higher learning rate and larger batch size helped the model converge better but may have caused overfitting on some classes, leading to worse performance on WATER"
]

In [60]:
trials[2].observations = [
    "Accuracy of CEMENT class improved significantly but all other classes got worse"
]

trials[2].conclusions = [
    "Not using oversampling caused the model to overfit on the majority classes, leading to better performance on CEMENT but worse performance on minority classes like WATER"
]

In [76]:
trials[3].observations = [
    "Greatly reduced performance across SAND,WATER and GREENERY classes"
] 

trials[3].conclusions = [
    "Dropout after features caused underfitting and reduced the model's ability to learn important patterns, leading to worse performance across multiple classes"
]

In [78]:
trials[4].observations = [
    "More consistent performance across all classes but still not the best"
]

trials[4].conclusions = [
    "Combining Adam optimizer with oversampling provided a more balanced learning process, improving performance on minority classes while maintaining decent performance on majority classes, though further tuning may be needed to achieve the best results"
]

In [79]:
trials[-1].observations = [
    "The best trial achieved the highest accuracy and F1 across all classes"
]

trials[-1].conclusions = [
    "The combination of oversampling, Adam optimizer, and a moderate learning rate of 0.001 provided the best balance for this dataset, leading to superior performance across all macro-classes"
]

In [80]:
to_submit = [trial for trial in trials if trial.observations != [""] and trial.conclusions != [""]]

In [81]:
class LabSubmission(BaseModel):
    student_ids: List[str]
    trials: List[TrialRecord]
    micro_class_error_analysis: str

# The records are populated here based on experimental iterations.
trial_one = TrialRecord(
    hyperparams={"learning_rate": 0.001, "batch_size": 32, "optimizer": "Adam", "epochs": 10},
    accuracy=0.0,
    F_score_class={"GREENERY": 0.0, "SAND": 0.0, "WATER": 0.0, "CEMENT_INFRASTRUCTURE": 0.0},
    observations=["Initial training phase completed.", "Validation loss fluctuated."],
    conclusions=["A lower learning rate may be required.", "Further regularization is recommended."]
)

# Additional trials are to be added to the list below.
submission = LabSubmission(
    student_ids=["9220154", "9220393"],
    trials=to_submit,
    micro_class_error_analysis=interpretation.strip()
)

# The Pydantic model is serialized to JSON and saved for grading.
output_json = submission.model_dump_json(indent=4)
print(output_json)

with open("student_submission.json", "w") as f:
    f.write(output_json)


{
    "student_ids": [
        "9220154",
        "9220393"
    ],
    "trials": [
        {
            "hyperparams": {
                "learning_rate": 0.001,
                "batch_size": 32,
                "optimizer": "SGD",
                "epochs": 10,
                "imbalance_strategy": "oversampling"
            },
            "accuracy": 0.8152272727272727,
            "F_score_class": {
                "GREENERY": 0.7726971504307488,
                "SAND": 0.7001862197392924,
                "WATER": 0.715258855585831,
                "CEMENT": 0.8856601389766267
            },
            "observations": [
                "All classes have relatively low F1 scores"
            ],
            "conclusions": [
                "The learning rate of 0.001 combined with SGD caused underfitting"
            ]
        },
        {
            "hyperparams": {
                "learning_rate": 0.01,
                "batch_size": 64,
                "optimizer": "SGD",
         

### Submission Protocol

The final deliverable **must be a zipped archive** submitted via Google Classroom. 

* The archive must contain the executed Jupyter Notebook and the generated `student_submission.json` file.
* The zipped file must be explicitly named utilizing the display name of the submitter exactly as it appears on Google Classroom.
* **Failure to adhere to these archival and naming conventions disrupts automated grading pipelines and will result in direct grade penalties.**

---

### Grading Rubric (Total: Ten Marks)

The requirement submission is evaluated based on the following criteria:

* **Data Engineering and Macro-Class Mapping (Two Marks)**
  The dataset is correctly loaded, successfully mapped to the defined Enum macro-classes, and securely partitioned into distinct training, validation, and testing subsets without data leakage.

* **Architecture and Training Execution (Two Marks)**
  A functional Convolutional Neural Network is constructed with appropriate spatial dimension tracking, and the training loop successfully computes gradients and updates model weights.

* **Hyperparameter Tuning (Two Marks)**
  Systematic hyperparameter tuning is executed, and multiple experimental trials are accurately serialized within the final JSON payload.

* **Error Analysis and Custom Confusion Matrix (Two Marks)**
  A custom confusion matrix mapping true micro-classes against predicted macro-classes is successfully generated, and systematic misclassification patterns for the least accurate class are logically analyzed in the written submission.

* **Quantitative Evaluation and Peer Comparison (Two Marks)**
  The macro-averaged F-score from the final experimental trial in the submitted list is explicitly reported and analytically compared against peer performances to identify relative architectural strengths and areas for optimization.
